In [1]:
import numpy as np
import pandas as pd
import scipy
from scipy import stats
import suncalc

import dask.dataframe as dd
from pathlib import Path
from tqdm import tqdm
import re

import datetime as dt

import matplotlib.pyplot as plt
from matplotlib import colors
import soundfile as sf
import matplotlib.patches as patches

In [2]:
import sys

sys.path.append("../src")
sys.path.append("../src/activity")

In [3]:
import subsampling as ss
import activity_assembly as actvt
import bout.assembly as bt
import comparison.data_assembly as comp
import comparison.plot as complot
from core import SITE_NAMES

from cli import get_file_paths
import plot
import pipeline

In [4]:
avail = np.arange(0, 180, 2) + 2
reset_3 = avail[np.where((3*60 % avail) == 0)[0]]
reset_4 = avail[np.where((4*60 % avail) == 0)[0]]
reset_6 = avail[np.where((6*60 % avail) == 0)[0]]
reset_12 = avail[np.where((12*60 % avail) == 0)[0]]
reset_24 = avail[np.where((24*60 % avail) == 0)[0]]

In [5]:
dt_starts = {'Carp high':dt.datetime(2022, 7, 15, 3, 0, 0),
             'Carp low':dt.datetime(2022, 9, 15, 0, 0, 0),
           'Telephone high':dt.datetime(2022, 8, 20, 3, 0, 0),
           'Telephone low':dt.datetime(2022, 9, 17, 0, 0, 0),
             'Central':dt.datetime(2022, 7, 10, 0, 0, 0),
             'Foliage':dt.datetime(2022, 7, 10, 0, 0, 0)}
dt_ends = {'Carp high':dt.datetime(2022, 8, 15, 13, 0, 0),
           'Carp low':dt.datetime(2022, 10, 15, 0, 0, 0),
           'Telephone high':dt.datetime(2022, 9, 20, 13, 0, 0),
           'Telephone low':dt.datetime(2022, 10, 17, 13, 0, 0),
           'Central':dt.datetime(2022, 10, 30, 16, 0, 0),
           'Foliage':dt.datetime(2022, 10, 30, 16, 0, 0)}
high_activity_types = {'Carp':'LF', 'Telephone':'HF'}

In [6]:
step = 1/10
step_by = np.arange(step, 1, step)
# step_by = step_by[step_by>=(1/6)]
step_by

array([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9])

In [7]:
step = 1/6
step_by = np.arange(0, (2/3)+step, step)
step_by = step_by[step_by>=(1/6)]
step_by

array([0.16666667, 0.33333333, 0.5       , 0.66666667])

In [8]:
cycle_lengths = [6, 10, 30, 48, 60, 72, 90]
percent_ons = step_by
data_params = dict()
data_params["cycle_lengths"] = cycle_lengths
data_params["percent_ons"] = percent_ons
dc_tags = ss.get_list_of_dc_tags(cycle_lengths, percent_ons)
dc_tags

['30of30',
 '1of6',
 '2of6',
 '3of6',
 '4of6',
 '2of10',
 '3of10',
 '5of10',
 '7of10',
 '5of30',
 '10of30',
 '15of30',
 '20of30',
 '8of48',
 '16of48',
 '24of48',
 '32of48',
 '10of60',
 '20of60',
 '30of60',
 '40of60',
 '12of72',
 '24of72',
 '36of72',
 '48of72',
 '15of90',
 '30of90',
 '45of90',
 '60of90']

In [9]:
site_keys = ['Carp', 'Central', 'Foliage', "Telephone"]
type_keys = ['LF', 'HF']
data_params["dc_tags"] = dc_tags
data_params['cur_dc_tag'] = '30of30'
data_params['recording_start'] = '00:00'
data_params['recording_end'] = '16:00'
data_params['assembly_type'] = 'kmeans'

In [ ]:
for site_key in site_keys:
    for type_key in type_keys:
        print(site_key, type_key)
        data_params["site_tag"] = site_key
        data_params["site_name"] = SITE_NAMES[site_key]
        data_params["type_tag"] = type_key
        data_params["detector_tag"] = 'bd2'
        data_params['metric_tag'] = 'bout_time_percentage'
        file_paths = get_file_paths(data_params)

        activitybout_arr, btp_arr = comp.generate_activity_btp_for_false_positives_investigation(data_params, file_paths, save=True)

Carp LF


In [ ]:
for site_key in site_keys:
    for type_key in type_keys:
        print(site_key, type_key)
        data_params["site_tag"] = site_key
        data_params["site_name"] = SITE_NAMES[site_key]
        data_params["type_tag"] = type_key
        data_params["detector_tag"] = 'bd2'
        file_paths = get_file_paths(data_params)

        activitybout_arr, btp_arr = comp.generate_activity_btp_for_false_negatives_investigation(data_params, file_paths, save=True)

Carp LF


100%|██████████| 6/6 [00:44<00:00,  7.34s/it]


Carp HF


100%|██████████| 6/6 [00:12<00:00,  2.13s/it]


Central LF


...

100%|██████████| 6/6 [00:31<00:00,  5.24s/it]


Telephone LF


100%|██████████| 6/6 [00:03<00:00,  1.76it/s]


Telephone HF


100%|██████████| 6/6 [00:35<00:00,  5.91s/it]


In [ ]:
for site_key in site_keys:
    for type_key in type_keys:
        print(site_key, type_key)
        data_params["site_tag"] = site_key
        data_params["site_name"] = SITE_NAMES[site_key]
        data_params["type_tag"] = type_key
        data_params['metric_tag'] = 'call_rate'
        data_params["detector_tag"] = 'bd2'
        file_paths = get_file_paths(data_params)

        activitycallrate_arr, callrate_arr = comp.generate_activity_call_rate_for_false_positives_investigation(data_params, file_paths, save=True)

Carp LF


100%|██████████| 41/41 [02:54<00:00,  4.26s/it]


Carp HF


100%|██████████| 41/41 [01:50<00:00,  2.69s/it]


Central LF


...

100%|██████████| 41/41 [02:47<00:00,  4.09s/it]


Telephone LF


100%|██████████| 41/41 [00:37<00:00,  1.10it/s]


Telephone HF


100%|██████████| 41/41 [02:33<00:00,  3.74s/it]


In [ ]:
for site_key in site_keys:
    for type_key in type_keys:
        data_params['metric_tag'] = 'call_rate'
        print(site_key, type_key)
        data_params["site_tag"] = site_key
        data_params["site_name"] = SITE_NAMES[site_key]
        data_params["type_tag"] = type_key
        data_params["detector_tag"] = 'bd2'
        file_paths = get_file_paths(data_params)

        activitycallrate_arr, callrate_arr = comp.generate_activity_call_rate_for_false_negatives_investigation(data_params, file_paths, save=True)

Carp LF


100%|██████████| 6/6 [00:11<00:00,  1.98s/it]


Carp HF


100%|██████████| 6/6 [00:04<00:00,  1.41it/s]


Central LF


...

100%|██████████| 6/6 [00:08<00:00,  1.46s/it]


Telephone LF


100%|██████████| 6/6 [00:01<00:00,  3.34it/s]


Telephone HF


100%|██████████| 6/6 [00:11<00:00,  1.96s/it]


In [ ]:
for site_key in site_keys:
    for type_key in type_keys:
        print(site_key, type_key)
        data_params["site_tag"] = site_key
        data_params["site_name"] = SITE_NAMES[site_key]
        data_params["type_tag"] = type_key
        data_params["detector_tag"] = 'bd2'
        data_params['metric_tag'] = 'activity_index'
        data_params['index_time_block_in_secs'] = 5
        file_paths = get_file_paths(data_params)

        activityind_arr, actvtind_arr = comp.generate_activity_index_percent_for_false_positives_investigation(data_params, file_paths, save=True)

Carp LF


100%|██████████| 41/41 [03:03<00:00,  4.46s/it]


Carp HF


100%|██████████| 41/41 [01:54<00:00,  2.79s/it]


Central LF


...

100%|██████████| 41/41 [02:52<00:00,  4.22s/it]


Telephone LF


100%|██████████| 41/41 [00:38<00:00,  1.05it/s]


Telephone HF


100%|██████████| 41/41 [02:37<00:00,  3.84s/it]


In [ ]:
for site_key in site_keys:
    for type_key in type_keys:
        print(site_key, type_key)
        data_params["site_tag"] = site_key
        data_params["site_name"] = SITE_NAMES[site_key]
        data_params["type_tag"] = type_key
        data_params["detector_tag"] = 'bd2'
        file_paths = get_file_paths(data_params)

        activityind_arr, actvtind_arr = comp.generate_activity_index_percent_for_false_negatives_investigation(data_params, file_paths, save=True)

Carp LF


100%|██████████| 6/6 [00:12<00:00,  2.03s/it]


Carp HF


100%|██████████| 6/6 [00:04<00:00,  1.35it/s]


Central LF


...

100%|██████████| 6/6 [00:09<00:00,  1.54s/it]


Telephone LF


100%|██████████| 6/6 [00:01<00:00,  3.07it/s]


Telephone HF


100%|██████████| 6/6 [00:11<00:00,  2.00s/it]
